# LoRA SFT on Qwen2.5-1.5B-Instruct with retrieved facts in the prompt (TRL + PEFT, single T4)

Follow-up to `sft_lora_experiments_v2data.ipynb` (TinyLlama, no context). The end-to-end RAG test showed that the
TinyLlama SFT model does not use retrieved passages and answers worse with them. Two changes:

- **Model**: Qwen/Qwen2.5-1.5B-Instruct, which follows context far better than TinyLlama.
- **Data**: the same v2 questions and Gemini answers, but the router-"knowledge" rows carry the agent's own
  "Facts that may help" system prompt (built locally by `scripts/build_rag_sft.py`, 70% of those rows; the rest stay
  without facts). Targets are unchanged. Prompt format = the serving prompt.

**Setup**: upload `data/children_sft_v3_rag/{train,validation}.jsonl` to Drive at `DATA_DIR` below.
The hold-out is the HF v2 test split without facts (style/readability only). Whether facts reduce hallucination is
measured by the end-to-end test (`eval/rag_e2e.py` + the claim judge), not here.

- **Configs**: C (lr 1e-4, r=32, all linear, alpha 16) and E (same, alpha 64; best on TinyLlama)
- **Metrics**: BERTScore-F1, ROUGE-L, eval loss / token accuracy, FKGL gap to the reference age level
- Everything is saved to Drive; finished configs are skipped on re-run.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U trl peft evaluate bert_score rouge_score textstat
# Colab ships an old torchao that new PEFT rejects at import time; it is not needed here
!pip uninstall -y -q torchao
# If anything was imported before this cell, restart the session (Runtime -> Restart session) once

## 0. Config

In [ ]:
import os, json, time, gc, random
from collections import Counter
import numpy as np
import torch

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET = "yxpan/children_sft_dataset_v2"            # hold-out only (test split, no facts)
DATA_DIR = "/content/drive/MyDrive/llm_sft_data/children_sft_v3_rag"   # train / validation with facts
EXP_NAME = "sft_lora_qwen_rag"  # fixed name so a re-run resumes instead of starting over
SEED = 42

MAX_STEPS = 350               # ~2 epochs over the 700 v2 training rows at batch 4 (v1 runs used 256 for 500 rows)
BATCH_SIZE = 4
EVAL_STEPS = 32
MAX_LEN = 1024
N_HOLDOUT = 600               # v1 data only (row-index split); v2 uses its whole test split

# generation settings from the original experiment; seeded so runs are repeatable
GEN_KWARGS = dict(max_new_tokens=128, do_sample=True, temperature=0.8, top_p=0.9, top_k=30,
                  repetition_penalty=1.05)
GEN_BATCH = 16

ATTN_QV = ["q_proj", "v_proj"]
ALL_LINEAR = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
CONFIGS = {
    "C":        dict(lr=1e-4, r=32, alpha=16, targets=ALL_LINEAR),
    "E_alpha64": dict(lr=1e-4, r=32, alpha=64, targets=ALL_LINEAR),   # LoRA scale alpha/r: 0.5 -> 2
}
BEST = None                   # None = pick the SFT config with the highest eval BERT-F1

import transformers
transformers.logging.set_verbosity_error()   # hide the repeated max_length / max_new_tokens notice

In [ ]:
# Save to Drive on Colab, to $OUT_ROOT elsewhere
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = f"/content/drive/MyDrive/llm_sft_runs/{EXP_NAME}"
    TMP_DIR = "/content/tmp_sft"
except ImportError:
    OUT_DIR = os.path.join(os.environ.get("OUT_ROOT", os.path.expanduser("~/llm_sft_runs")), EXP_NAME)
    TMP_DIR = "tmp_sft"
os.makedirs(OUT_DIR, exist_ok=True)

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
import transformers, trl, peft
ENV = dict(gpu=torch.cuda.get_device_name(0), torch=torch.__version__,
           transformers=transformers.__version__, trl=trl.__version__, peft=peft.__version__)
save_json(dict(env=ENV, configs=CONFIGS, max_steps=MAX_STEPS, batch_size=BATCH_SIZE,
               gen_kwargs=GEN_KWARGS, seed=SEED), f"{OUT_DIR}/setup.json")
print(ENV); print("Outputs ->", OUT_DIR)

## 1. Data

In [ ]:
from datasets import load_dataset

def read_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]

# train / validation: v3 rows with a ready-made `messages` field [system (+facts), user, assistant]
from datasets import Dataset
train_rows = read_jsonl(f"{DATA_DIR}/train.jsonl")
EVAL_ROWS = read_jsonl(f"{DATA_DIR}/validation.jsonl")
holdout_ds = load_dataset(DATASET)["test"]          # no facts: readability / style check only
HOLDOUT_ROWS = list(holdout_ds)
for name, d in [("train", train_rows), ("eval", EVAL_ROWS), ("hold-out", HOLDOUT_ROWS)]:
    print(f"{name:>8}: {len(d):4d} rows, ages {dict(sorted(Counter(r['age'] for r in d).items()))}")
print("with facts: train", sum(r["with_facts"] for r in train_rows), "eval", sum(r["with_facts"] for r in EVAL_ROWS))

def system_prompt(age):
    return (f"You are talking to a child aged {age}. "
            f"Generate a friendly response with age-appropriate knowledge.")

def prompt_messages(row):
    if "messages" in row:                            # v3 rows: the exact serving prompt
        return row["messages"][:-1]
    return [{"role": "system", "content": system_prompt(row["age"])},
            {"role": "user", "content": row["instruction"]}]

# prompt-completion format: loss is computed on the assistant answer only
def to_prompt_completion(row):
    return {"prompt": row["messages"][:-1], "completion": [row["messages"][-1]]}

train_pc = Dataset.from_list([to_prompt_completion(r) for r in train_rows])
eval_pc = Dataset.from_list([to_prompt_completion(r) for r in EVAL_ROWS])
print(train_pc[0]["prompt"][0]["content"][:400])
train_pc[0]


## 2. Evaluation helpers

- **BERTScore-F1 / ROUGE-L**: model answer vs reference answer
- **Age alignment**: the reference answers were written for each child's age, so their readability is the target.
  `fkgl_gap` = mean |FKGL(model answer) - FKGL(reference)| in grade levels (lower is better);
  `fkgl_within2` = share of answers within 2 grade levels of the reference (higher is better).

In [ ]:
import evaluate, textstat
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

@torch.no_grad()
def generate(model, rows):
    tok.padding_side = "left"
    torch.manual_seed(SEED)
    preds = []
    for i in range(0, len(rows), GEN_BATCH):
        texts = [tok.apply_chat_template(prompt_messages(r), tokenize=False, add_generation_prompt=True)
                 for r in rows[i:i + GEN_BATCH]]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        out = model.generate(**enc, **GEN_KWARGS, pad_token_id=tok.pad_token_id)
        preds += tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    tok.padding_side = "right"
    return [p.strip() for p in preds]

def fkgl(text):
    return textstat.flesch_kincaid_grade(text) if text.strip() else np.nan

def text_metrics(preds, rows):
    refs = [r["response"] for r in rows]
    bert = bertscore.compute(predictions=preds, references=refs, lang="en", batch_size=32)["f1"]
    gap = np.abs(np.array([fkgl(p) for p in preds]) - np.array([fkgl(t) for t in refs]))
    return dict(bert_f1=float(np.mean(bert)),
                rougeL=float(rouge.compute(predictions=preds, references=refs)["rougeL"]),
                fkgl_gap=float(np.nanmean(gap)),
                fkgl_within2=float(np.nanmean(gap <= 2)),
                empty_frac=float(np.mean([not p for p in preds])))

def per_knowledge(preds, rows):
    if "needs_knowledge" not in rows[0]:
        return {}
    out = {}
    for flag, label in [(True, "knowledge"), (False, "non-knowledge")]:
        idx = [i for i, r in enumerate(rows) if bool(r["needs_knowledge"]) == flag]
        out[label] = dict(n=len(idx), **text_metrics([preds[i] for i in idx], [rows[i] for i in idx]))
    return out

def per_age(preds, rows):
    out = {}
    for age in sorted(set(r["age"] for r in rows)):
        idx = [i for i, r in enumerate(rows) if r["age"] == age]
        out[age] = dict(n=len(idx), **text_metrics([preds[i] for i in idx], [rows[i] for i in idx]))
    return out

def save_preds(preds, rows, path):
    with open(path, "w") as f:
        for p, r in zip(preds, rows):
            f.write(json.dumps(dict(age=r["age"], instruction=r["instruction"],
                                    reference=r["response"], prediction=p)) + "\n")

def load_base():
    return AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16).to("cuda").eval()

def free(*objs):
    for o in objs:
        del o
    gc.collect(); torch.cuda.empty_cache()

## 3. Base model (no SFT) on the eval set

In [ ]:
base_path = f"{OUT_DIR}/base/metrics.json"
if os.path.exists(base_path):
    print("skip, already done"); BASE_METRICS = json.load(open(base_path))
else:
    os.makedirs(f"{OUT_DIR}/base", exist_ok=True)
    model = load_base()
    preds = generate(model, EVAL_ROWS)
    BASE_METRICS = dict(config="base (no SFT)", **text_metrics(preds, EVAL_ROWS))
    save_preds(preds, EVAL_ROWS, f"{OUT_DIR}/base/eval_preds.jsonl")
    save_json(BASE_METRICS, base_path)
    free(model)
print(BASE_METRICS)

## 4. Train the four LoRA configs

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

def train_one(name, cfg):
    run_dir = f"{OUT_DIR}/{name}"
    adapter_file = f"{run_dir}/adapter/adapter_model.safetensors"
    if (os.path.exists(f"{run_dir}/metrics.json") and os.path.exists(adapter_file)
            and os.path.getsize(adapter_file) > 0):   # both, in case a disconnect lost the adapter upload
        print(f"[{name}] skip, already done")
        return json.load(open(f"{run_dir}/metrics.json"))
    os.makedirs(run_dir, exist_ok=True)

    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16).to("cuda")  # T4 has no bf16; if loss turns NaN, load in float32
    peft_cfg = LoraConfig(r=cfg["r"], lora_alpha=cfg["alpha"], lora_dropout=0.1,
                          target_modules=cfg["targets"], bias="none", task_type="CAUSAL_LM")
    args = SFTConfig(output_dir=f"{TMP_DIR}/{name}", learning_rate=cfg["lr"], lr_scheduler_type="linear",
                     per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
                     max_steps=cfg.get("steps", MAX_STEPS), logging_steps=8, eval_strategy="steps", eval_steps=EVAL_STEPS,
                     save_strategy="no", fp16=True, bf16=False, max_length=MAX_LEN, seed=SEED, report_to="none")
    trainer = SFTTrainer(model=model, args=args, train_dataset=train_pc, eval_dataset=eval_pc,
                         processing_class=tok, peft_config=peft_cfg)
    n_trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)

    t0 = time.time(); trainer.train(); train_min = (time.time() - t0) / 60
    ev = trainer.evaluate()
    trainer.model.save_pretrained(f"{run_dir}/adapter")
    tok.save_pretrained(f"{run_dir}/adapter")
    save_json(trainer.state.log_history, f"{run_dir}/log_history.json")

    m = trainer.model.eval(); m.config.use_cache = True
    preds = generate(m, EVAL_ROWS)
    save_preds(preds, EVAL_ROWS, f"{run_dir}/eval_preds.jsonl")
    metrics = dict(config=name, lr=cfg["lr"], r=cfg["r"], alpha=cfg["alpha"], steps=cfg.get("steps", MAX_STEPS),
                   targets="all linear" if cfg["targets"] == ALL_LINEAR else ",".join(cfg["targets"]),
                   trainable_params=n_trainable, train_min=round(train_min, 2),
                   eval_loss=ev.get("eval_loss"), eval_token_acc=ev.get("eval_mean_token_accuracy"),
                   **text_metrics(preds, EVAL_ROWS))
    save_json(metrics, f"{run_dir}/metrics.json")
    free(trainer, model, m)
    print(f"[{name}]", metrics)
    return metrics

RESULTS = {name: train_one(name, cfg) for name, cfg in CONFIGS.items()}

## 5. Compare configs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

table = pd.DataFrame([BASE_METRICS] + list(RESULTS.values())).set_index("config")
cols = ["lr", "r", "alpha", "steps", "targets", "trainable_params", "train_min", "eval_loss", "eval_token_acc",
        "bert_f1", "rougeL", "fkgl_gap", "fkgl_within2", "empty_frac"]
table = table[[c for c in cols if c in table.columns]]
display(table.round(4))
table.to_csv(f"{OUT_DIR}/config_comparison.csv")

if BEST is None:
    BEST = max(RESULTS, key=lambda k: RESULTS[k]["bert_f1"])
print("best config:", BEST)
save_json(dict(best=BEST, selected_by="eval bert_f1"), f"{OUT_DIR}/best_config.json")  # read by the GRPO notebook

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name in CONFIGS:
    hist = json.load(open(f"{OUT_DIR}/{name}/log_history.json"))
    tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
    ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
    axes[0].plot(*zip(*tr), label=name, alpha=0.8)
    axes[1].plot(*zip(*ev), "o-", label=name)
axes[0].set_title("train loss"); axes[1].set_title("eval loss")
for ax in axes[:2]:
    ax.set_xlabel("step"); ax.legend()
names = list(table.index)
x = np.arange(len(names))
axes[2].bar(x - 0.2, table["bert_f1"], 0.4, label="BERT-F1")
axes[2].bar(x + 0.2, table["fkgl_within2"], 0.4, label="within 2 grades")
axes[2].set_xticks(x, names, rotation=20); axes[2].set_ylim(0, 1); axes[2].legend()
axes[2].set_title("eval set (100)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/config_comparison.png", dpi=150); plt.show()

## 6. Final test: base vs best config on the hold-out set

The hold-out rows were never used for training or for choosing the config.

In [ ]:
from peft import PeftModel

final_path = f"{OUT_DIR}/holdout_{BEST}.json"
if os.path.exists(final_path):
    print("skip, already done"); FINAL = json.load(open(final_path))
else:
    FINAL = {}
    model = load_base()
    preds = generate(model, HOLDOUT_ROWS)
    save_preds(preds, HOLDOUT_ROWS, f"{OUT_DIR}/base/holdout_preds.jsonl")
    FINAL["base"] = dict(overall=text_metrics(preds, HOLDOUT_ROWS), per_age=per_age(preds, HOLDOUT_ROWS),
                         per_knowledge=per_knowledge(preds, HOLDOUT_ROWS))
    save_json(FINAL, final_path + ".partial")

    model = PeftModel.from_pretrained(model, f"{OUT_DIR}/{BEST}/adapter").eval()
    preds = generate(model, HOLDOUT_ROWS)
    save_preds(preds, HOLDOUT_ROWS, f"{OUT_DIR}/{BEST}/holdout_preds.jsonl")
    FINAL[BEST] = dict(overall=text_metrics(preds, HOLDOUT_ROWS), per_age=per_age(preds, HOLDOUT_ROWS),
                       per_knowledge=per_knowledge(preds, HOLDOUT_ROWS))
    save_json(FINAL, final_path)
    free(model)

b, s = FINAL["base"]["overall"], FINAL[BEST]["overall"]
summary = pd.DataFrame({"base": b, BEST: s}).T
summary.loc["relative change"] = [(s[k] - b[k]) / b[k] if b[k] else np.nan for k in summary.columns]
display(summary.round(4))
summary.to_csv(f"{OUT_DIR}/holdout_summary.csv")

rows = []
for age in FINAL["base"]["per_age"]:
    rb, rs = FINAL["base"]["per_age"][age], FINAL[BEST]["per_age"][age]
    rows.append(dict(age=age, n=rb["n"], bert_f1_base=rb["bert_f1"], bert_f1_sft=rs["bert_f1"],
                     fkgl_gap_base=rb["fkgl_gap"], fkgl_gap_sft=rs["fkgl_gap"],
                     within2_base=rb["fkgl_within2"], within2_sft=rs["fkgl_within2"]))
by_age = pd.DataFrame(rows).set_index("age")
display(by_age.round(3))
by_age.to_csv(f"{OUT_DIR}/holdout_by_age.csv")

if FINAL["base"].get("per_knowledge"):
    by_kn = pd.DataFrame([dict(group=g, n=FINAL["base"]["per_knowledge"][g]["n"],
                               **{f"{m}_{who}": FINAL[key]["per_knowledge"][g][m]
                                  for m in ("bert_f1", "fkgl_gap", "fkgl_within2")
                                  for who, key in (("base", "base"), ("sft", BEST))})
                          for g in FINAL["base"]["per_knowledge"]]).set_index("group")
    display(by_kn.round(3))
    by_kn.to_csv(f"{OUT_DIR}/holdout_by_knowledge.csv")

## 7. Adapter for the serving benchmark

In [ ]:
ADAPTER_PATH = f"{OUT_DIR}/{BEST}/adapter"
print("Set ADAPTER_PATH in vllm_serving_benchmark.ipynb to:\n ", ADAPTER_PATH)
# Optional: push to the Hugging Face Hub so other notebooks / machines can load it by name
# from huggingface_hub import login; login()
# PeftModel.from_pretrained(load_base(), ADAPTER_PATH).push_to_hub("yxpan/qwen2.5-1.5b-child-sft-lora")